In [ ]:
import mdtraj as md
import MDAnalysis as mda

from sys import stdout

# OpenMM imports
import openmm.app as app
import openmm as mm
import openmm.unit as unit
from openmmforcefields.generators import SMIRNOFFTemplateGenerator

# OpenFF-toolkit imports
from openff.toolkit import Molecule
from openff.toolkit import Topology as offTopology
from openff.units.openmm import to_openmm as offquantity_to_openmm

import parmed as pmd

import subprocess
from collections import defaultdict
from pathlib import Path

In [2]:
def PBSA_calc(traj_file, top_file, ligand_path, cond):
    traj = md.load(traj_file, top=top_file)

    # remove all other atoms except protein
    complex_atoms = traj.topology.select('chainid 0 or resname UNK')
    traj_complex = traj.atom_slice(complex_atoms)
    traj_complex.save('traj_complex.dcd')
    traj_complex[0].save('traj_complex.pdb')

    protein_atoms = traj.topology.select('chainid 0')
    traj_protein = traj.atom_slice(protein_atoms)
    traj_protein[0].save('traj_protein.pdb')

    ligand_atoms = traj.topology.select('resname UNK')
    traj_ligand = traj.atom_slice(ligand_atoms)
    traj_ligand[0].save('traj_ligand.pdb')

    complex_pdb = app.PDBFile('traj_complex.pdb')
    protein_pdb = app.PDBFile('traj_protein.pdb')
    ff = app.ForceField('amber/protein.ff14SB.xml', 'amber/tip3p_standard.xml')

    ligand = Molecule.from_file(ligand_path)
    smirnoff = SMIRNOFFTemplateGenerator(molecules=ligand)
    ff.registerTemplateGenerator(smirnoff.generator)

    ligand_off_topology = offTopology.from_molecules(molecules=[ligand])
    ligand_omm_topology = ligand_off_topology.to_openmm()
    ligand_positions = offquantity_to_openmm(ligand.conformers[0])

    modeller = app.Modeller(protein_pdb.topology, protein_pdb.positions)

    modeller.add(ligand_omm_topology, ligand_positions)

    system = ff.createSystem(modeller.topology)

    structure = pmd.openmm.load_topology(complex_pdb.topology, system=system, xyz=complex_pdb.positions)
    structure.save('complex.prmtop', overwrite=True)

    rec_atoms = traj.topology.select('chainid 0')  # chain A
    rec_traj = traj.atom_slice(rec_atoms)
    rec_traj[0].save('receptor.pdb')
    rec_pdb = app.PDBFile('receptor.pdb')
    modeller = app.Modeller(rec_pdb.topology, rec_pdb.positions)
    system = ff.createSystem(modeller.topology)
    rec_struct = pmd.openmm.load_topology(rec_pdb.topology, system=system, xyz=rec_pdb.positions)
    rec_struct.save('receptor.prmtop', overwrite=True)

    complex = pmd.load_file("complex.prmtop")
    complex.strip("!:UNK") 
    complex.save("ligand.prmtop", overwrite=True)

    infile = "mmpbsa.in"

    mmpbsa_content = """&general
    startframe=51,
    endframe=250,
    interval=5,
    verbose=1,
    /
    &decomp
    idecomp=1,
    dec_verbose=1,
    /
    &pb
    istrng=0.15,
    /
    """

    with open(infile, "w+") as f:
        f.write(mmpbsa_content)

    cmd = [
        "MMPBSA.py",
        "-O",
        "-i", "mmpbsa.in",
        "-cp", "complex.prmtop",
        "-rp", "receptor.prmtop",
        "-lp", "ligand.prmtop",
        "-y", "traj_complex.dcd",
        "-o", f"output/ligand_o_{cond}.dat",
        "-eo", f"output/ligand_eo_{cond}.dat",
        "-do", f"output/ligand_do_{cond}.dat",
        "-deo", f"output/ligand_deo_{cond}.dat",
    ]

    subprocess.run(cmd, capture_output=True, text=True)

In [ ]:
def strip_name(name, pre, suf):
    if pre and name.startswith(pre):
        name = name[len(pre):]

    if suf and name.endswith(suf):
        name = name[:-len(suf)]

    return name

def pdb_to_sdf(pdb_path, sdf_path):
    cmd = [
        "obabel",
        pdb_path,
        "-O", sdf_path,
    ]
    subprocess.run(cmd, check=True)

In [6]:
topformat = 'pdb'
trajformat = 'dcd'
inputf = 'input'
folder = Path(inputf)

top_ext = "." + topformat.lower().lstrip(".")
traj_ext = "." + trajformat.lower().lstrip(".")

pairs = defaultdict(dict)

for f in folder.iterdir():
    ext = f.suffix.lower()

    if ext == top_ext:
        core = strip_name(f.stem, '', '_topology_processed')
        pairs[core]["top"] = f

    elif ext == traj_ext:
        core = strip_name(f.stem, '', '_traj_processed')
        pairs[core]["traj"] = f

valid_pairs = []

for name, files in pairs.items():
    if "top" in files and "traj" in files:
        valid_pairs.append((files["top"], files["traj"]))

In [ ]:
for sample in valid_pairs:
    u = mda.Universe(sample[0])
    ligand = u.select_atoms("segid X and resname UNK")
    ligand.write(f"input/ligand.pdb")
    pdb_to_sdf(f"input/ligand.pdb", f"input/ligand.sdf")
    
    filename = strip_name(str(sample[0]), 'input/', '_topology_processed.pdb')
    #PBSA_calc('input/' + sample[1], 'input/' + sample[0], 'input_backup/ligand.sdf', filename)

fex_10_1
fex_10_2
fex_19_1
fex_19_2
fex_8_1
fex_8_2
osu_1_1
osu_1_2
osu_2_1
osu_2_2
osu_5_1
osu_5_2
